# Case study: leaderboard round L2 with gradient boosting

*Session 10, block 3 practice. Author: course team, licence CC-BY-4.0.*

**Task.** Predict the four-digit HS heading of each test decision with a **tree-based model on engineered dense features**, compare it honestly with the linear text model, look inside it, and write the submission file for round L2.

Trees cannot work directly on a TF-IDF matrix with hundreds of thousands of sparse columns. We therefore compress the description into **150 dense text components** (character n-gram TF-IDF followed by truncated SVD, a PCA for sparse matrices; Sessions 11 and 13) and add the leakage-free features of Session 9: text statistics, language, issuing country and date. Inputs that the test set does not have (`keywords`, `classification_justification`, `cn_code`, `chapter`, `status`, `end_date`) are never used.

Plan:
1. Data and a time-based validation split (fit 2017–2021, validate 2022–2023), as on the leaderboard.
2. Features.
3. The current model: a linear classifier on the sparse TF-IDF matrix (the L3 method of Session 13, here as the benchmark).
4. A pitfall of multiclass boosting with many rare classes.
5. The challenger: LightGBM on the dense features.
6. Comparison with a bootstrap interval, and permutation importance.
7. Submission.

Runtime: about 10 minutes on a laptop (LightGBM fits one tree per heading and round). On macOS LightGBM needs `brew install libomp`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
import time

import lightgbm
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score

## 1. Data and time-based split

In [ ]:
sample = pd.read_parquet(DATA / "train_sample.parquet")
test = pd.read_parquet(DATA / "test.parquet")
fit = (sample["start_date"] < "2022-01-01").to_numpy()      # 2017-2021
val = ~fit                                                    # 2022-2023
y = sample["heading"].to_numpy()
print(fit.sum(), val.sum(), len(test))
print("headings in the sample:", sample["heading"].nunique(),
      "| validation headings never seen in fitting:", round(np.mean(~np.isin(y[val], y[fit])), 4))

## 2. Features

**Text components.** The vectoriser and the SVD are fitted on the fitting rows only (2017–2021) and then applied to the validation and test rows.

In [ ]:
t0 = time.time()
vectoriser = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=100_000,
                             sublinear_tf=True, dtype=np.float32)
X_fit_sparse = vectoriser.fit_transform(sample.loc[fit, "description"])
X_val_sparse = vectoriser.transform(sample.loc[val, "description"])
svd = TruncatedSVD(n_components=150, random_state=0).fit(X_fit_sparse)
print(X_fit_sparse.shape, "explained variance of 150 components:", round(svd.explained_variance_ratio_.sum(), 3),
      f"({time.time() - t0:.0f} s)")

**Metadata and text statistics** (Session 9). Language and issuing country are passed as pandas categories, which LightGBM splits on natively (page 3).

In [ ]:
def text_stats(description):
    """Simple statistics of the description of goods (Session 9, page 2), one row per decision."""
    d = description.fillna("")
    n_chars = d.str.len()
    return pd.DataFrame({
        "log_chars": np.log1p(n_chars),
        "n_digits": d.str.count(r"\d"),
        "n_lines": d.str.count("\n") + 1,
        "upper_share": d.str.count(r"[A-ZÄÖÜ]") / n_chars.clip(lower=1),
        "mean_word_len": n_chars / d.str.split().str.len().clip(lower=1),
        "has_code": d.str.contains("<CODE>", regex=False).astype(int),
    }, index=description.index)


def dense_features(frame, components):
    out = pd.DataFrame(components, index=frame.index, columns=[f"svd{i:03d}" for i in range(components.shape[1])])
    out = out.join(text_stats(frame["description"]))
    out["language"] = pd.Categorical(frame["language"], categories=sorted(sample["language"].unique()))
    out["country"] = pd.Categorical(frame["issuing_country"], categories=sorted(sample["issuing_country"].unique()))
    out["year"] = frame["start_date"].dt.year
    out["month"] = frame["start_date"].dt.month
    return out


F_fit = dense_features(sample[fit], svd.transform(X_fit_sparse))
F_val = dense_features(sample[val], svd.transform(X_val_sparse))
F_fit.iloc[:2, -10:]

**Question.** `year` is a legitimate input (the test set has `start_date`), but trees cannot extrapolate it: what will the model do with the test years 2024–2026?

## 3. The current model: linear classifier on the sparse TF-IDF matrix

In [ ]:
t0 = time.time()
current = SGDClassifier(loss="hinge", alpha=1e-5, random_state=0, n_jobs=-1).fit(X_fit_sparse, y[fit])
pred_current = current.predict(X_val_sparse)
print(f"linear on TF-IDF: accuracy {accuracy_score(y[val], pred_current):.3f}, "
      f"macro-F1 {f1_score(y[val], pred_current, average='macro'):.3f} ({time.time() - t0:.0f} s)")

## 4. A pitfall: Newton steps for rare classes

A multiclass booster fits one tree per class and round. In a leaf with few rows of a rare class, the hessian (second derivative of the loss) is almost zero, and the leaf value gradient / hessian explodes. We show it on the coarser task with 97 chapters, which is faster.

In [ ]:
for min_hessian in (1e-3, 1.0):                  # 1e-3 is LightGBM's default
    t0 = time.time()
    m = lightgbm.LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, min_child_samples=20,
                                min_sum_hessian_in_leaf=min_hessian, verbose=-1, random_state=0)
    m.fit(F_fit.filter(like="svd"), sample.loc[fit, "chapter"])
    acc = accuracy_score(sample.loc[val, "chapter"], m.predict(F_val.filter(like="svd")))
    print(f"min_sum_hessian_in_leaf={min_hessian}: chapter accuracy {acc:.3f} ({time.time() - t0:.0f} s)")

## 5. The challenger: LightGBM on the dense features (1,114 headings)

In [ ]:
t0 = time.time()
challenger = lightgbm.LGBMClassifier(
    n_estimators=150, learning_rate=0.1, num_leaves=15, min_child_samples=10,
    min_sum_hessian_in_leaf=1.0, colsample_bytree=0.5, verbose=-1, random_state=0)
challenger.fit(F_fit, y[fit])
print(f"fitted in {time.time() - t0:.0f} s")
for rounds in (25, 50, 100, 150):                  # accuracy as trees are added
    p = challenger.predict(F_val, num_iteration=rounds)
    print(rounds, round(accuracy_score(y[val], p), 3), round(f1_score(y[val], p, average="macro"), 3))
pred_challenger = challenger.predict(F_val)

## 6. Comparison and interpretation

### 6.1 Paired comparison with a bootstrap interval

In [ ]:
correct_cur = pred_current == y[val]
correct_ch = pred_challenger == y[val]
rng = np.random.default_rng(0)
diffs = [np.mean(correct_ch[i] - correct_cur[i].astype(float))
         for i in (rng.integers(0, val.sum(), val.sum()) for _ in range(1000))]
chapter_of = lambda h: np.array([s[:2] for s in h])
summary = pd.DataFrame({
    "accuracy": [correct_cur.mean(), correct_ch.mean()],
    "macro-F1": [f1_score(y[val], pred_current, average="macro"), f1_score(y[val], pred_challenger, average="macro")],
    "chapter accuracy": [np.mean(chapter_of(pred_current) == chapter_of(y[val])),
                         np.mean(chapter_of(pred_challenger) == chapter_of(y[val]))],
}, index=["current: linear on TF-IDF", "challenger: LightGBM on dense features"])
print("difference challenger - current, 95 % bootstrap interval:", np.percentile(diffs, [2.5, 97.5]).round(3))
summary.round(3)

**Question.** Which model would you submit if the leaderboard were the only criterion? Which would you deploy, given training time, prediction time and the number of trees (1,114 headings × 150 rounds)?

### 6.2 Permutation importance by feature group

The 150 text components only make sense together, so we permute them as one block; each metadata column is permuted alone. We use 3,000 validation decisions to keep it quick.

In [ ]:
sub = F_val.sample(3000, random_state=0)
y_sub = pd.Series(y[val], index=F_val.index).loc[sub.index].to_numpy()
base = accuracy_score(y_sub, challenger.predict(sub))
groups = {"text components (150)": list(F_val.filter(like="svd").columns),
          **{c: [c] for c in ["log_chars", "n_digits", "n_lines", "upper_share", "mean_word_len",
                              "has_code", "language", "country", "year", "month"]}}
rows = []
for name, cols in groups.items():
    shuffled = sub.copy()
    perm = rng.permutation(len(sub))
    for c in cols:
        shuffled[c] = sub[c].to_numpy()[perm]
    if "language" in cols or "country" in cols:
        for c in cols:
            shuffled[c] = pd.Categorical(shuffled[c], categories=sub[c].cat.categories)
    rows.append({"group": name, "drop in accuracy": base - accuracy_score(y_sub, challenger.predict(shuffled))})
print("accuracy on the 3,000 rows:", round(base, 3))
pd.DataFrame(rows).sort_values("drop in accuracy", ascending=False).round(3)

**Reading.** Almost everything comes from the text components; the metadata add little. SHAP values (page 4) would give 1,114 sets of contributions per decision, one per heading; for a model with this many classes, permutation importance on groups is the more useful summary.

## 7. Submission for round L2

We refit the challenger on the whole sample (2017–2023), keeping the vectoriser and the SVD fitted on 2017–2021, and predict the 113,188 test decisions.

In [ ]:
t0 = time.time()
F_all = pd.concat([F_fit, F_val]).loc[sample.index]
final = lightgbm.LGBMClassifier(**challenger.get_params()).fit(F_all, y)
F_test = dense_features(test, svd.transform(vectoriser.transform(test["description"])))
submission = pd.DataFrame({"id": test["id"], "heading": final.predict(F_test)})
submission.to_csv("submission_l2.csv", index=False)
print(submission.shape, submission["heading"].str.fullmatch(r"\d{4}").all(), f"({time.time() - t0:.0f} s)")
submission.head()

Score the file from the repository root (if your course provides the scorer) and submit it:

```bash
uv run python case-study/score.py sessions/10-tree-based-models/workbooks/submission_l2.csv
```

## Exercises

1. **Hierarchical model.** Train LightGBM for the 97 chapters, then a separate linear model per chapter for the headings. Does the chapter-first approach beat the flat challenger?
2. **More components.** Repeat with 300 SVD components. How do accuracy and training time change?
3. **Stacking.** Add the linear model's decision score for its top heading as a feature of the booster (computed out-of-fold!). Why does it have to be out-of-fold?
4. Write three sentences for the team's model log: what the challenger is, how it compares with the current model (with the interval), and your decision.